In [1]:
p_bronze_account = "saedmsprdizadls01"
p_bronze_container_src = "caas-edms-bronze/air transport/ati/raw/"
p_bronze_container_dst = "caas-edms-bronze/air transport/ati/raw/"
p_bronze_container_arch = "caas-edms-bronze/air transport/ati/archive/"
p_source_name = "ATI_City Links"
p_dynamic_data_range = 1
p_silver_schema_mapping = '''{
            "type": "TabularTranslator",
            "mappings": [
                {
                    "source": {
                        "name": "Airport Code",
                        "type": "String",
                        "physicalType": "String"
                    },
                    "sink": {
                        "name": "Airport_Code",
                        "type": "String",
                        "physicalType": "UTF8"
                    }
                },
                {
                    "source": {
                        "name": "Country Name",
                        "type": "String",
                        "physicalType": "String"
                    },
                    "sink": {
                        "name": "Country_Name",
                        "type": "String",
                        "physicalType": "UTF8"
                    }
                },
                {
                    "source": {
                        "name": "City Name",
                        "type": "String",
                        "physicalType": "String"
                    },
                    "sink": {
                        "name": "City_Name",
                        "type": "String",
                        "physicalType": "UTF8"
                    }
                },
                {
                    "source": {
                        "name": "PaxCarrier(s)",
                        "type": "String",
                        "physicalType": "String"
                    },
                    "sink": {
                        "name": "PaxCarrier",
                        "type": "String",
                        "physicalType": "UTF8"
                    }
                },
                {
                    "source": {
                        "name": "PaxCarrier(s) Count",
                        "type": "String",
                        "physicalType": "String"
                    },
                    "sink": {
                        "name": "PaxCarrier_Count",
                        "type": "String",
                        "physicalType": "UTF8"
                    }
                },
                {
                    "source": {
                        "name": "Pax for Cargo Carrier(s)",
                        "type": "String",
                        "physicalType": "String"
                    },
                    "sink": {
                        "name": "Pax_For_Cargo_Carrier",
                        "type": "String",
                        "physicalType": "UTF8"
                    }
                },
                {
                    "source": {
                        "name": "Pax for Cargo Carrier(s) Count",
                        "type": "String",
                        "physicalType": "String"
                    },
                    "sink": {
                        "name": "Pax_For_Cargo_Carrier_Count",
                        "type": "String",
                        "physicalType": "UTF8"
                    }
                },
                {
                    "source": {
                        "name": "Frt Carrier(s)",
                        "type": "String",
                        "physicalType": "String"
                    },
                    "sink": {
                        "name": "Frt_Carrier",
                        "type": "String",
                        "physicalType": "UTF8"
                    }
                },
                {
                    "source": {
                        "name": "Frt Carrier(s) Count",
                        "type": "String",
                        "physicalType": "String"
                    },
                    "sink": {
                        "name": "Frt_Carrier_Count",
                        "type": "String",
                        "physicalType": "UTF8"
                    }
                },
                {
                    "source": {
                        "name": "OpsType_Pax",
                        "type": "String",
                        "physicalType": "String"
                    },
                    "sink": {
                        "name": "OpsType_Pax",
                        "type": "String",
                        "physicalType": "UTF8"
                    }
                },
                {
                    "source": {
                        "name": "OpsType_Frt",
                        "type": "String",
                        "physicalType": "String"
                    },
                    "sink": {
                        "name": "OpsType_Frt",
                        "type": "String",
                        "physicalType": "UTF8"
                    }
                },
                {
                    "source": {
                        "name": "OpsType_Codeshare",
                        "type": "String",
                        "physicalType": "String"
                    },
                    "sink": {
                        "name": "OpsType_Codeshare",
                        "type": "String",
                        "physicalType": "UTF8"
                    }
                },
                {
                    "source": {
                        "name": "Region",
                        "type": "String",
                        "physicalType": "String"
                    },
                    "sink": {
                        "name": "Region",
                        "type": "String",
                        "physicalType": "UTF8"
                    }
                }'''
p_silver_mandatory_columns='Airport_Code,Country_Name,City_Name'
secret_name = "scr-edms-decrypt-pwd"

In [2]:
import os
import io
import re
import json
import pandas as pd
import openpyxl
from notebookutils import mssparkutils 
from azure.storage.blob import BlobServiceClient, ContainerClient
password = bytes(mssparkutils.credentials.getSecret("kv-edms-prdiz-001",secret_name,"ls_kv_edms_prd01"), 'ascii')
accountkey = mssparkutils.credentials.getSecret("kv-edms-prdiz-001","scr-edms-saedmsprdizadls01-key","ls_kv_edms_prd01")
conn_str = "DefaultEndpointsProtocol=https;AccountName={0};AccountKey={1};EndpointSuffix=core.windows.net".format(p_bronze_account,accountkey)

### Helpers

In [19]:
def extract_source_columns_from_mapping(schema_mapping_str):
    if not schema_mapping_str:
        return None
    try:
        mapping_str = schema_mapping_str.strip()
        try:
            mapping_json = json.loads(mapping_str)
        except json.JSONDecodeError:
            for fix in ["}", "]}", "}]}"]:
                try:
                    mapping_json = json.loads(mapping_str + fix)
                    print(f"  Fixed incomplete JSON with: '{fix}'")
                    break
                except json.JSONDecodeError:
                    continue
            else:
                raise ValueError("Could not repair JSON after all fix attempts.")
        source_cols = [m["source"]["name"] for m in mapping_json["mappings"]]
        print(f"  Source columns from schema mapping: {source_cols}")
        return source_cols
    except Exception as e:
        print(f"  Warning: Could not parse silver_schema_mapping — {e}. Falling back to auto-detect.")
        return None

def extract_mandatory_columns(mandatory_columns_str, schema_mapping_str):
    if not mandatory_columns_str:
        return []
    mandatory_sink_names = [col.strip() for col in mandatory_columns_str.split(",") if col.strip()]
    if not schema_mapping_str:
        return mandatory_sink_names
    try:
        mapping_str = schema_mapping_str.strip()
        try:
            mapping_json = json.loads(mapping_str)
        except json.JSONDecodeError:
            for fix in ["}", "]}", "}]}"]:
                try:
                    mapping_json = json.loads(mapping_str + fix)
                    break
                except json.JSONDecodeError:
                    continue
            else:
                return mandatory_sink_names
        # Build sink -> source lookup
        sink_to_source = {
            m["sink"]["name"]: m["source"]["name"]
            for m in mapping_json["mappings"]
        }
        resolved = []
        for sink_name in mandatory_sink_names:
            if sink_name in sink_to_source:
                resolved.append(sink_to_source[sink_name])
                print(f"Mandatory column resolved: '{sink_name}' -> '{sink_to_source[sink_name]}'")
            else:
                print(f"Warning: mandatory column '{sink_name}' not found in schema mapping, using as-is.")
                resolved.append(sink_name)
        return resolved
    except Exception as e:
        print(f"Warning: Could not resolve mandatory columns — {e}. Using as-is.")
        return mandatory_sink_names

def parse_headers(row):
    seen = {}
    headers = []
    for i, cell in enumerate(row):
        if cell is None or str(cell).strip() == "":
            name = f"Unnamed: {i}"
        else:
            name = str(cell).strip()
        if name in seen:
            seen[name] += 1
            name = f"{name}.{seen[name]}"
        else:
            seen[name] = 0
        headers.append(name)
    return headers

def is_schema_header_row(row, source_columns, match_threshold=0.6):
    row_vals = [str(cell).strip() for cell in row if cell is not None and str(cell).strip() != ""]
    if not row_vals:
        return False
    matches = sum(1 for col in source_columns if col in row_vals)
    return (matches / len(source_columns)) >= match_threshold

def is_repeat_header(row, canonical_headers, match_threshold=0.6):
    canonical_filled = [(i, h) for i, h in enumerate(canonical_headers) if not h.startswith("Unnamed: ")]
    if not canonical_filled:
        return False
    matches = sum(
        1 for i, h in canonical_filled
        if i < len(row) and row[i] is not None and str(row[i]).strip() == h
    )
    return (matches / len(canonical_filled)) >= match_threshold

def get_mandatory_col_indices(headers, mandatory_columns):
    # Returns list of column indices for mandatory columns found in headers
    indices = []
    for col in mandatory_columns:
        if col in headers:
            indices.append(headers.index(col))
        else:
            print(f"  Warning: mandatory column '{col}' not found in headers, skipping.")
    return indices

def is_mandatory_empty(row, mandatory_col_indices):
    # Row is considered end-of-table if ANY mandatory column is empty
    for idx in mandatory_col_indices:
        val = row[idx] if len(row) > idx else None
        if val is None or str(val).strip() == "":
            return True
    return False

def read_sheet_all_tables(ws, source_columns=None, mandatory_columns=None):
    rows = list(ws.iter_rows(values_only=True))
    total_rows = len(rows)
    tables = []
    canonical_headers = None
    mandatory_col_indices = []
    scan_idx = 0

    while scan_idx < total_rows:
        # Find next header row
        header_row_idx = None
        for i in range(scan_idx, total_rows):
            row = rows[i]
            if source_columns is not None:
                if is_schema_header_row(row, source_columns, match_threshold=0.6):
                    header_row_idx = i
                    print(f"  Header found at row {i+1} via schema mapping match.")
                    break
            else:
                if canonical_headers is None:
                    if is_header_row(row, ws.max_column):
                        header_row_idx = i
                        print(f"  Header found at row {i+1} via fill ratio.")
                        break
                else:
                    if is_repeat_header(row, canonical_headers, match_threshold=0.6):
                        header_row_idx = i
                        break
        if header_row_idx is None:
            break

        headers = parse_headers(rows[header_row_idx])

        if canonical_headers is None:
            canonical_headers = headers
            if mandatory_columns:
                mandatory_col_indices = get_mandatory_col_indices(headers, mandatory_columns)
                print(f"  Mandatory column indices: {mandatory_col_indices}")
            if not mandatory_col_indices:
                # Fallback: use first non-Unnamed header column
                for idx, h in enumerate(headers):
                    if not h.startswith("Unnamed: "):
                        mandatory_col_indices = [idx]
                        print(f"  No mandatory columns resolved, falling back to first header column: '{h}' (index {idx})")
                        break

        data = []
        i = header_row_idx + 1
        while i < total_rows:
            row = rows[i]

            # Check if this row is the start of a repeated/next table header
            is_next_table = (
                is_schema_header_row(row, source_columns, match_threshold=0.6)
                if source_columns is not None
                else (canonical_headers and is_repeat_header(row, canonical_headers, match_threshold=0.6))
            )
            if is_next_table:
                scan_idx = i
                break

            # Stop if any mandatory column is empty
            if is_mandatory_empty(row, mandatory_col_indices):
                scan_idx = i + 1
                break

            data.append(dict(zip(headers, row)))
            i += 1
        else:
            scan_idx = total_rows

        if data:
            tables.append((headers, data))
        else:
            scan_idx = header_row_idx + 1

    return tables

def excel_blob_to_dataframes(blob_content, source_columns=None, mandatory_columns=None):
    wb = openpyxl.load_workbook(io.BytesIO(blob_content), data_only=True)
    result = {}
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        tables = read_sheet_all_tables(ws, source_columns=source_columns, mandatory_columns=mandatory_columns)
        if not tables:
            print(f"  [{sheet_name}] No tables detected, skipping sheet.")
            continue
        if p_dynamic_data_range == 2:
            tables = tables[:1]
            print(f"  [{sheet_name}] dynamic_data_range=2 — extracting first table only.")
        reference_columns = None
        all_dfs = []
        for idx, (headers, data) in enumerate(tables, start=1):
            df = pd.DataFrame(data, columns=headers)
            df = df.fillna("")
            df = df[~df.apply(lambda row: all(str(v).strip() == "" for v in row), axis=1)]
            if reference_columns is None:
                reference_columns = df.columns.tolist()
            rename_map = {
                col: reference_columns[i]
                for i, col in enumerate(df.columns)
                if col.startswith("Unnamed: ") and i < len(reference_columns)
            }
            df = df.rename(columns=rename_map).reindex(columns=reference_columns)
            all_dfs.append(df)
        if all_dfs:
            combined_df = pd.concat(all_dfs, ignore_index=True)
            result[sheet_name] = combined_df
            print(f"  [{sheet_name}] Combined → {len(combined_df)} total rows")
    return result

### XLSX

In [24]:
source_columns = extract_source_columns_from_mapping(p_silver_schema_mapping)
mandatory_columns = extract_mandatory_columns(p_silver_mandatory_columns, p_silver_schema_mapping)
blob_service_client = BlobServiceClient.from_connection_string(conn_str)
p_bronze_container = p_bronze_container_src.split("/", 1)[0]
p_bronze_container_src_path = p_bronze_container_src.split("/", 1)[-1]
src_container_client = blob_service_client.get_container_client(p_bronze_container)

for blob in src_container_client.list_blobs():
    if p_bronze_container_src_path in blob.name:
        if blob.name.endswith('.xlsx'):
            blob_filename = blob.name.split("/")[-1]
            if not blob_filename.startswith(p_source_name):
                print(f"Skipping {blob_filename} — does not match source: {p_source_name}")
                continue
            blob_client = blob_service_client.get_blob_client(p_bronze_container, blob.name)
            print(f"Processing {blob.name}...")
            blob_content = blob_client.download_blob().readall()
            dataframes = excel_blob_to_dataframes(blob_content, source_columns=source_columns, mandatory_columns=mandatory_columns)
            if not dataframes:
                print(f"No tables detected in {blob.name}, skipping.")
                continue
            temp_file_path = "/tmp/temp.xlsx"
            with pd.ExcelWriter(temp_file_path, engine='openpyxl') as writer:
                for sheet_name, df in dataframes.items():
                    df.to_excel(writer, sheet_name=sheet_name, index=False)
            with open(temp_file_path, "rb") as f:
                blob_client.upload_blob(f, overwrite=True)
            print(f"Saved to {p_bronze_container_dst}")
        else:
            print(f"{blob.name} is not an xlsx file. Skipping.")

### XLSM

In [54]:
source_columns = extract_source_columns_from_mapping(p_silver_schema_mapping)
mandatory_columns = extract_mandatory_columns(p_silver_mandatory_columns, p_silver_schema_mapping)
blob_service_client = BlobServiceClient.from_connection_string(conn_str)
p_bronze_container = p_bronze_container_src.split("/", 1)[0]
p_bronze_container_src_path = p_bronze_container_src.split("/", 1)[-1]
src_container_client = blob_service_client.get_container_client(p_bronze_container)

for blob in src_container_client.list_blobs():
    if p_bronze_container_src_path in blob.name:
        if blob.name.endswith('.xlsm'):
            blob_filename = blob.name.split("/")[-1]
            if not blob_filename.startswith(p_source_name):
                print(f"Skipping {blob_filename} — does not match source: {p_source_name}")
                continue
            blob_client = blob_service_client.get_blob_client(p_bronze_container, blob.name)
            print(f"Processing {blob.name}...")
            blob_content = blob_client.download_blob().readall()
            print("Process dynamic")
            dataframes = excel_blob_to_dataframes(blob_content, source_columns=source_columns, mandatory_columns=mandatory_columns)
            if not dataframes:
                print(f"No tables detected in {blob.name}, skipping.")
                continue
            temp_file_path = "/tmp/temp.xlsx"
            with pd.ExcelWriter(temp_file_path, engine='openpyxl') as writer:
                for sheet_name, df in dataframes.items():
                    df.to_excel(writer, sheet_name=sheet_name, index=False)
            if os.path.exists(temp_file_path):
                with open(temp_file_path, "rb") as f:
                    file_content = f.read()
                    new_blob_name = blob.name.replace('.xlsm', '.xlsx')
                    new_blob_client = blob_service_client.get_blob_client(p_bronze_container, new_blob_name)
                    new_blob_client.upload_blob(file_content, overwrite=True)
                print(f"  Saved as {new_blob_name}")
                blob_client.delete_blob()
                print(f"  Original {blob.name} deleted.")
            else:
                print("  Failed to save temp file.")
        else:
            print(f"{blob.name} is not an xlsm file. Skipping.")